## Bronze ingestion: orchestration
This notebook builds parameters, calls the main ingestion function, 
and shows the result in the Databricks UI.


In [0]:
import sys, os
sys.path.append(os.path.abspath(".."))

# Clear stale partial imports from previous failed runs
for _m in list(sys.modules):
    if _m.startswith("src."):
        del sys.modules[_m]

# from src.cdse_client import get_cdse_access_token
from src.control_table import (
    ensure_control_schema_exists,
    get_already_ingested,
)
from src.ingestion import run_ingestion
import yaml

In [0]:
def find_repo_root(start_dir: str, marker_relative_path: str = "conf/ingestion_config.yaml", max_levels: int = 8) -> str:
    current = start_dir
    for _ in range(max_levels):
        candidate = os.path.join(current, marker_relative_path)
        if os.path.exists(candidate):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise FileNotFoundError(
        f"Could not find '{marker_relative_path}' coming from {start_dir}"
    )

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

notebook_fs_path = f"/Workspace{notebook_path}" if not notebook_path.startswith("/Workspace") else notebook_path
start_dir = os.path.dirname(notebook_fs_path)

repo_root = find_repo_root(start_dir)
config_path = f"{repo_root}/conf/ingestion_config.yaml"

dbutils.widgets.text("max_cloud_cover_override", "")
max_cloud_cover_override = dbutils.widgets.get("max_cloud_cover_override")

with open(config_path) as f:
    config = yaml.safe_load(f)

max_cloud_cover = (
    float(max_cloud_cover_override)
    if max_cloud_cover_override
    else config["ingestion"]["max_cloud_cover_pct"]
)

In [0]:
# Credentials
username = dbutils.secrets.get(scope=config["cdse"]["secret_scope"], key="username")
password = dbutils.secrets.get(scope=config["cdse"]["secret_scope"], key="password")


In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

uc = config["unity_catalog"]

control_table = f"{uc['catalog']}.{uc['control_schema']}.ingestion_log"

ensure_control_schema_exists(spark, uc["catalog"], uc["control_schema"])
already_ingested = get_already_ingested(spark, control_table)

In [0]:
# token = get_cdse_access_token(
#     username=username,
#     password=password,
#     token_url=config["cdse"]["token_url"],
# )

In [0]:
from functools import partial
from src.cdse_client import get_cdse_access_token
from src.control_table import flush_log_records, get_run_id, make_log_record
from typing import Callable

get_token = partial(
    get_cdse_access_token,
    username=username,
    password=password,
    token_url=config["cdse"]["token_url"],
)
    

In [0]:
# dbutils.library.restartPython()


In [0]:
print(f"active_aoi: {config['stac_discovery']['active_aoi']}")
print(f"max_cloud_cover: {max_cloud_cover}")
print(already_ingested.count())

In [0]:

run_ingestion(
    spark=spark,
    config=config,
    get_token=get_token,
    already_ingested=already_ingested,
    control_table=control_table,
    max_cloud_cover=max_cloud_cover,
)



The end.